# Where could c14_age_bp, c14_error, d13C, pMC_value, pMC_error, cal_68 and cal_95 live in SEAD_staging?

This notebook explores the SEAD_staging schema for possible homes for these Strucke columns:
- dedicated columns created specifically for one of these values
- generic "measurement/value" tables where such a value could be stored via a foreign key (e.g. tied to a method, entity or dataset), rather than a purpose-built column


In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


## Connect to sead_staging database


In [2]:
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

DB_HOST = os.environ["DB_HOST"]
DB_PORT = os.environ["DB_PORT"]
DB_NAME = os.environ["DB_NAME"]
DB_USER = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")


## List all tables in the public schema


In [3]:
tables = pd.read_sql(
    "select table_name from information_schema.tables where table_schema = 'public' order by table_name",
    engine,
)
print(f'{len(tables)} tables/views in the public schema')
tables


181 tables/views in the public schema


,table_name
0,master_set_reference
1,taxon_view
2,tbl_abundance_elements
3,tbl_abundance_ident_levels
4,tbl_abundance_modifications
...,...
176,view_typed_analysis_tables
177,view_typed_analysis_values
178,view_with_abundances
179,view_with_references


## Search for columns whose name hints at these values

Keywords: age, error, d13c, delta, pmc, cal, c14, carbon, isotope, uncertain, value.


In [4]:
candidate_columns = pd.read_sql(
    text(
        """
        select table_name, column_name, data_type
        from information_schema.columns
        where table_schema = 'public'
          and (
            column_name ilike '%age%' or
            column_name ilike '%error%' or
            column_name ilike '%d13c%' or
            column_name ilike '%delta%' or
            column_name ilike '%pmc%' or
            column_name ilike '%cal%' or
            column_name ilike '%c14%' or
            column_name ilike '%carbon%' or
            column_name ilike '%isotope%' or
            column_name ilike '%uncertain%' or
            column_name ilike '%value%'
          )
        order by table_name, ordinal_position
        """
    ),
    engine,
)
print(f"{len(candidate_columns)} candidate columns across {candidate_columns['table_name'].nunique()} tables")
candidate_columns


176 candidate columns across 69 tables


,table_name,column_name,data_type
0,tbl_abundance_properties,property_value,text
1,tbl_age_types,age_type_id,integer
2,tbl_age_types,age_type,character varying
3,tbl_aggregate_sample_ages,aggregate_sample_age_id,integer
4,tbl_aggregate_sample_ages,analysis_entity_age_id,integer
...,...,...,...
171,view_with_abundances,locality,text
172,view_with_abundances,language,text
173,view_with_times,age_older,numeric
174,view_with_times,age_younger,numeric


### `tbl_geochronology` looks like the dedicated home for c14_age_bp, c14_error and d13C

It has `age`, `error_older`/`error_younger` and `delta_13c` columns, plus `dating_lab_id` and `lab_number` (matching Strucke's `lab_no`) and `analysis_entity_id` linking it to a physical sample.

No column anywhere in the schema is named for pMC or for a calibrated 68%/95% range, though — worth keeping in mind while reading the rest of this notebook.


In [5]:
geochronology_schema = pd.read_sql(
    "select column_name, data_type from information_schema.columns where table_name = 'tbl_geochronology' order by ordinal_position",
    engine,
)
geochronology_schema


,column_name,data_type
0,geochron_id,integer
1,analysis_entity_id,bigint
2,dating_lab_id,integer
3,lab_number,character varying
4,age,numeric
5,error_older,numeric
6,error_younger,numeric
7,delta_13c,numeric
8,notes,text
9,date_updated,timestamp with time zone


In [6]:
geochronology = pd.read_sql('select * from public.tbl_geochronology', engine)
print(f'{len(geochronology)} rows in tbl_geochronology')
print(f"{geochronology['delta_13c'].notna().sum()} rows have delta_13c populated")
geochronology[['geochron_id', 'dating_lab_id', 'lab_number', 'age', 'error_older', 'error_younger', 'delta_13c']].sample(10, random_state=1)


1463 rows in tbl_geochronology
0 rows have delta_13c populated


,geochron_id,dating_lab_id,lab_number,age,error_older,error_younger,delta_13c
719,720,774,K-4819,3780.0,85.0,85.0,None
683,684,883,UB-4581,7894.0,35.0,35.0,None
503,504,863,SRR-3461,11060.0,70.0,70.0,None
424,424,916,HUTH-3212,167500.0,5400.0,3500.0,None
846,845,743,GrN-18157,26430.0,240.0,240.0,None
860,858,743,GrN-18149,24590.0,120.0,120.0,None
1078,1080,702,Birm-409,42000.0,1000.0,1000.0,None
1080,1082,702,Birm-409,42000.0,1000.0,1000.0,None
1132,1134,700,Beta-316485,6250.0,40.0,40.0,None
895,893,687,AAR-1279,6900.0,100.0,100.0,None


## The generic analysis_entity → analysis_value pattern

SEAD also has a generic EAV-style chain for measurements that don't get a purpose-built column:

`tbl_datasets` → `tbl_dataset_methods` (which `tbl_methods` was used) → `tbl_analysis_entities` (one per physical_sample+dataset) → `tbl_analysis_values` (one row per measured property, tagged with a `value_class_id`) → a typed subtype table holding the actual value: `tbl_analysis_numerical_values`, `tbl_analysis_integer_values`, `tbl_analysis_categorical_values`, `tbl_analysis_boolean_values`, `tbl_analysis_dating_ranges`, `tbl_analysis_numerical_ranges`, `tbl_analysis_integer_ranges`, `tbl_analysis_identifiers` or `tbl_analysis_notes`.

If any of our target columns are stored generically rather than in a dedicated column, this is the mechanism that would hold them, tied together via foreign keys rather than a named column.


In [7]:
c14_methods = pd.read_sql(
    text(
        """
        select method_id, method_name, method_abbrev_or_alt_name, method_group_id
        from tbl_methods
        where method_name ilike '%radiocarbon%'
           or method_name ilike '%c14%'
           or method_name ilike '%carbon%'
           or method_name ilike '%calib%'
        order by method_id
        """
    ),
    engine,
)
c14_methods


,method_id,method_name,method_abbrev_or_alt_name,method_group_id
0,38,C14 Accelerator dating,C14 AMS,3
1,39,C14 Conventional,C14,3
2,129,Archaeological period C14 years,ArchPerC14,19
3,132,Geological C14 period,GeolPerC14,19
4,136,Tephrochronology C14,TephraC14,20
5,148,Radiocarbon (14C Unspecified),C14 Unspec.,3
6,151,C14 Conventional,C14 Std,3
7,153,C14 dating of humic substances in sediment,C14 Humous,3
8,156,Calibrated radiocarbon date (method unspecified),Cal,20
9,157,Calibrated AMS radiocarbon date,CalAMS,20


### Are any of these methods actually used by a dataset?

If a method is never referenced in `tbl_dataset_methods`, the generic value tables can't currently hold anything tagged with it.


In [8]:
method_ids = ','.join(str(m) for m in c14_methods['method_id'])
method_usage = pd.read_sql(
    f"""
    select method_id, count(*) as dataset_count
    from tbl_dataset_methods
    where method_id in ({method_ids})
    group by method_id
    """,
    engine,
)
print(f"{len(method_usage)} of the {len(c14_methods)} C14-related methods are referenced in tbl_dataset_methods")
method_usage


0 of the 11 C14-related methods are referenced in tbl_dataset_methods


,method_id,dataset_count


### Row counts across the generic value-storage chain


In [9]:
generic_value_tables = pd.read_sql(
    """
    select 'tbl_dataset_methods' as table_name, count(*) as row_count from tbl_dataset_methods
    union all select 'tbl_analysis_entities', count(*) from tbl_analysis_entities
    union all select 'tbl_analysis_values', count(*) from tbl_analysis_values
    union all select 'tbl_analysis_numerical_values', count(*) from tbl_analysis_numerical_values
    union all select 'tbl_analysis_integer_values', count(*) from tbl_analysis_integer_values
    union all select 'tbl_analysis_dating_ranges', count(*) from tbl_analysis_dating_ranges
    union all select 'tbl_analysis_numerical_ranges', count(*) from tbl_analysis_numerical_ranges
    union all select 'tbl_analysis_integer_ranges', count(*) from tbl_analysis_integer_ranges
    """,
    engine,
)
generic_value_tables


,table_name,row_count
0,tbl_analysis_numerical_values,60
1,tbl_analysis_numerical_ranges,0
2,tbl_analysis_integer_ranges,0
3,tbl_dataset_methods,0
4,tbl_analysis_dating_ranges,7775
5,tbl_analysis_integer_values,25790
6,tbl_analysis_values,72183
7,tbl_analysis_entities,163168


`tbl_dataset_methods` being empty means the generic pathway isn't currently wired up to any radiocarbon method in this staging copy of the database, even though `tbl_analysis_dating_ranges` and `tbl_analysis_values` already hold rows for other purposes. So today, `tbl_geochronology` is the only place that actually holds C14 age/error/d13C data — the generic tables are a structural possibility for the future, not a current data source to reconcile against.


## Isotope-specific generic tables (another possible home for d13C)

Independently of `tbl_geochronology.delta_13c`, SEAD has a generic isotope-measurement pattern: `tbl_isotopes` (one row per measurement) → `tbl_isotope_measurements` → `tbl_isotope_types` (a full periodic table of elements, so "carbon" is just one of many) and `tbl_isotope_standards`.


In [10]:
isotope_tables = pd.read_sql(
    """
    select 'tbl_isotopes' as table_name, count(*) as row_count from tbl_isotopes
    union all select 'tbl_isotope_measurements', count(*) from tbl_isotope_measurements
    union all select 'tbl_isotope_standards', count(*) from tbl_isotope_standards
    """,
    engine,
)
isotope_tables


,table_name,row_count
0,tbl_isotopes,0
1,tbl_isotope_measurements,0
2,tbl_isotope_standards,1


## Units that might hint at pMC or calibrated-year storage

If pMC values were stored anywhere, we would expect a matching unit definition.


In [11]:
units = pd.read_sql('select unit_id, unit_name, unit_abbrev from public.tbl_units order by unit_id', engine)
print(f'{len(units)} units defined')
units


15 units defined


,unit_id,unit_name,unit_abbrev
0,1,metres,m
1,2,kilograms,kg
2,3,litres,l
3,4,Decimal degrees,dd
4,5,Units,NaN
5,6,Astronomical unit,NaN
6,7,14C years,C14yrs
7,8,Years,yrs
8,9,Degrees Celcius,°C
9,10,millimetres,mm


## Summary

- **c14_age_bp → `tbl_geochronology.age`**, **c14_error → `error_older`/`error_younger`**, **d13C → `delta_13c`**. This table already holds 1,463 real radiocarbon dates (recognisable lab codes like Ua-, GrN-, OxA-, Birm-), and also carries `lab_number` and `dating_lab_id`, matching Strucke's `lab_no`. `delta_13c` is present as a column but rarely populated in the sample checked.
- **pMC_value / pMC_error → no home found.** No column, method, or unit anywhere in the schema references percent modern carbon.
- **cal_68 / cal_95 → no dedicated home found either.** `tbl_analysis_dating_ranges` and `tbl_analysis_numerical_ranges` are structurally capable of holding a low/high range, and `tbl_methods` even has explicit "Calibrated radiocarbon date" method entries (156, 157) — but `tbl_dataset_methods` currently has zero rows for any C14-related method, so nothing is actually wired up to store a calibrated range today.
- The **generic analysis_entity → analysis_value chain** and the **generic isotope-measurement chain** are both real, working mechanisms elsewhere in SEAD, so they remain the most likely place pMC/cal ranges would go **if** they get modeled in the future — just not populated for radiocarbon today.
